# Getting Started with LangChain for Agents

The following brings in some content from the notebooks in the LangChain intro courses, e.g., as can be found at https://github.com/langchain-ai/lca-lc-foundations

For brief review, we can use LangChain's OpenAI connector to query against the NRP API.

In [1]:
import os
NRP_TOK = os.environ.get('NRP_TOK')
nrp_llm_url = "https://ellm.nrp-nautilus.io/v1"

In [2]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model = 'gpt-oss',
                   api_key = NRP_TOK,
                   base_url = nrp_llm_url,
                   # use_responses_api=False forces the classic chat.completions endpoint
                   use_responses_api=False,
                   temperature = 0.3)

After initializing our chat model, getting a response can be as easy as passing a string into the `invoke` method.

In [3]:
response = model.invoke('What is the capital of france?')

This has wrapped the model call and given us the response object, which contains the text output to our question plus a lot more.

In [4]:
response

AIMessage(content='The capital of France is **Paris**. It’s not only the political center of the country but also a major cultural, artistic, and economic hub known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre‑Dame Cathedral.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 72, 'total_tokens': 158, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-94a5ee26-6a00-41e7-9c05-8ac3850fc040', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d25-ec9b-7d80-bb7c-e50f9bd972d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 86, 'total_tokens': 158, 'input_token_details': {}, 'output_token_details': {}})

To specifically get the model's answer to our query:

In [5]:
response.content

'The capital of France is **Paris**. It’s not only the political center of the country but also a major cultural, artistic, and economic hub known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre‑Dame Cathedral.'

Other pieces of the response provide useful context on the metadata level:

In [6]:
response.response_metadata

{'token_usage': {'completion_tokens': 86,
  'prompt_tokens': 72,
  'total_tokens': 158,
  'completion_tokens_details': None,
  'prompt_tokens_details': None},
 'model_provider': 'openai',
 'model_name': 'openai/gpt-oss-120b',
 'system_fingerprint': None,
 'id': 'chatcmpl-94a5ee26-6a00-41e7-9c05-8ac3850fc040',
 'finish_reason': 'stop',
 'logprobs': None}

In [7]:
response.usage_metadata

{'input_tokens': 72,
 'output_tokens': 86,
 'total_tokens': 158,
 'input_token_details': {},
 'output_token_details': {}}

In [8]:
for i in response:
    print(f"{i[0]:>20} : {i[1]}\n")

             content : The capital of France is **Paris**. It’s not only the political center of the country but also a major cultural, artistic, and economic hub known for landmarks such as the Eiffel Tower, the Louvre Museum, and Notre‑Dame Cathedral.

   additional_kwargs : {'refusal': None}

   response_metadata : {'token_usage': {'completion_tokens': 86, 'prompt_tokens': 72, 'total_tokens': 158, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-94a5ee26-6a00-41e7-9c05-8ac3850fc040', 'finish_reason': 'stop', 'logprobs': None}

                type : ai

                name : None

                  id : lc_run--019c5d25-ec9b-7d80-bb7c-e50f9bd972d0-0

          tool_calls : []

  invalid_tool_calls : []

      usage_metadata : {'input_tokens': 72, 'output_tokens': 86, 'total_tokens': 158, 'input_token_details': {}, 'output_token_details': {}}



# Another simpler approach

In [9]:
from langchain.chat_models import init_chat_model

In [10]:
model = init_chat_model(model="gpt-oss",
                        base_url=nrp_llm_url,
                        api_key=NRP_TOK)

In [11]:
model.invoke("What is the capital of CA?")

AIMessage(content='The capital of **California (CA)** is **Sacramento**.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 72, 'total_tokens': 123, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-d380bc4d-9218-4a07-af43-34a6729dedd8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d26-29bd-7130-ab55-0ef9481e16db-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 51, 'total_tokens': 123, 'input_token_details': {}, 'output_token_details': {}})

# Creating an agent

In [12]:
from langchain.agents import create_agent

If you have an OPENAIKEY in your environment, you can do something like:
* `agent = create_agent('gpt-5')`

But here we need to use the chat model from above.

In [13]:
agent = create_agent(model=model)

There are a couple things we need to modify relative to invoking the model.

In [14]:
# This will give an ERROR
agent.invoke("What is the capital of CA?")

InvalidUpdateError: Expected dict, got What is the capital of CA?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

If you want to check, this will also not work:

In [15]:
# This will give an ERROR!
agent = create_agent(model="gpt-oss",
                     base_url=nrp_llm_url,
                     api_key=NRP_TOK)

TypeError: create_agent() got an unexpected keyword argument 'base_url'

We need to be a little careful in using a model with a different endpoint.  Here is a way to see some help documentation:

In [16]:
help(create_agent)

Help on function create_agent in module langchain.agents.factory:

create_agent(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | dict[str, Any] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache[Any] | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'
    Creates an agent graph that calls tools in a loop until a stopping condition is met.

    For more details on using `cre

A direct chat model instance (such as our ChatOpenAI instance above) is acceptable for the agent model.

In [17]:
model = ChatOpenAI(model = 'gpt-oss',
                 api_key = NRP_TOK,
                 base_url = nrp_llm_url,
                 # use_responses_api=False forces the classic chat.completions endpoint
                 use_responses_api=False,
                 temperature = 0.3)

In [18]:
agent = create_agent(model=model)

As far as `agent.invoke`, let's look at help while we're at it:

In [19]:
help(agent.invoke)

Help on method invoke in module langgraph.pregel.main:

invoke(input: 'InputT | Command | None', config: 'RunnableConfig | None' = None, *, context: 'ContextT | None' = None, stream_mode: 'StreamMode' = 'values', print_mode: 'StreamMode | Sequence[StreamMode]' = (), output_keys: 'str | Sequence[str] | None' = None, interrupt_before: 'All | Sequence[str] | None' = None, interrupt_after: 'All | Sequence[str] | None' = None, durability: 'Durability | None' = None, **kwargs: 'Any') -> 'dict[str, Any] | Any' method of langgraph.graph.state.CompiledStateGraph instance
    Run the graph with a single input and config.

    Args:
        input: The input data for the graph. It can be a dictionary or any other type.
        config: The configuration for the graph run.
        context: The static context to use for the run.
            !!! version-added "Added in version 0.6.0"
        stream_mode: The stream mode for the graph run.
        print_mode: Accepts the same values as `stream_mode`, b

Notice that there's a lot of mention of "graphs".  This is because we're working with LangGraph too.  

The `create_agent` function from `langchain.agents` is built on top of LangGraph, which provides the underlying graph-based execution engine for agents.

Our book has an example with the following:
* `from langchain.agents import AgentExecutor, create_react_agent`

I mentioned previously this needed to be changed to:
* `from langchain_classic.agents import AgentExecutor, create_react_agent`

We will stick to `langchain.agents.create_agent` but revisit the book example.

LangChain has been undergoing lots of modifications since the book came out.
* "LangChain and LangGraph Agent Frameworks Reach v1.0 Milestones": https://blog.langchain.com/langchain-langgraph-1dot0/
* LangGraph vs LangChain vs DeepAgents: https://docs.langchain.com/oss/python/concepts/products

If you turn to other tutorials, you may also see a variety of other frameworks pop up, including LlamaIndex and smolagents.  (Also LangSmith and LangServe -- these are frameworks useful for productionizing work)

| Dimension                               | LangChain                                                                                         | LangGraph                                                                                                 | LlamaIndex                                                                                       | smolagents                                                                                                           |
| --------------------------------------- | ------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------ | -------------------------------------------------------------------------------------------------------------------- |
| **Quick mental model**                  | General LLM app & agent framework. Gives you lots of building blocks (chains, tools, agents).   | Workflow / state machine for agents. You draw the graph of steps and LangGraph runs it.                 | Data / RAG framework. Focused on connecting LLMs to your own data (docs, DBs, etc.).           | "Tiny agent library." Lets models write & run Python code to do things, with minimal abstractions.                   |
| **Main focus**                          | Build LLM‑powered apps and agents with many integrations (models, vector DBs, tools).             | Build stateful, controllable, often multi‑step or multi‑agent workflows over time.                      | Make it easy to ingest, index, retrieve, and query your private data using LLMs.                 | Minimal code‑centric agents that call tools and execute code, often via sandboxes.                                   |
| **Good first use case**       | Simple tool‑using chatbot (e.g., question answering + calling a calculator or search API).        | A multi‑step pipeline: plan -> retrieve data -> call tool -> summarize, with clear steps in a graph.         | "Chat with your PDFs / SQL DB" RAG assistant over a dataset.                              | A research or coding helper that writes small Python snippets to answer questions or analyze data.                   |
| **Agent style**                         | Provides built‑in agent types (e.g., ReAct‑style) that decide when to call tools, how often, etc. | You explicitly define nodes (functions / LLM calls) and edges; the system handles control flow and state. | Data tools as first‑class citizens: query engines, indexes, and agents that use them as tools. | Code‑first agents: the model often responds with code that gets executed, plus simple tool‑calling variants.         |
| **Data / RAG support**                  | Good RAG support (loaders, retrievers, vector stores), but it’s one part of a broader framework.  | Not a data library by itself; usually wraps RAG steps you build with LangChain or custom tools.           | RAG‑first: indexing, retrieval, query orchestration, and evaluation are core to the library.     | No special RAG layer; you wrap search / DB / RAG systems as tools the agent can use.                                 |

# Back to work

Our previous work with roles (e.g. system, user) as elements of model messages take on a new flavor.

In the above you might have noticed `AIMessage`, and we can also work with `SystemMessage` and `HumanMessage`

In [20]:
from langchain.messages import HumanMessage, SystemMessage, AIMessage

Now we get to a working `agent.invoke`

In [21]:
response = agent.invoke(
    {'messages': [HumanMessage(content='What is the capital of france?')]}
)

In [22]:
response

{'messages': [HumanMessage(content='What is the capital of france?', additional_kwargs={}, response_metadata={}, id='83b6af2b-e952-4071-b741-356fa5f2f033'),
  AIMessage(content='The capital of France is **Paris**. It’s not only the political center but also a major hub for culture, art, fashion, and history.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 72, 'total_tokens': 138, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-5a23f60b-864f-4b45-825b-c29ac3c4d395', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d26-68c7-7383-bfac-b9033a8e29a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 66, 'total_tokens': 138, 'input_token_details': {}, 'output_token_details': {}})]}

In [23]:
response['messages'][0]

HumanMessage(content='What is the capital of france?', additional_kwargs={}, response_metadata={}, id='83b6af2b-e952-4071-b741-356fa5f2f033')

In [24]:
response['messages'][1]

AIMessage(content='The capital of France is **Paris**. It’s not only the political center but also a major hub for culture, art, fashion, and history.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 72, 'total_tokens': 138, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-5a23f60b-864f-4b45-825b-c29ac3c4d395', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d26-68c7-7383-bfac-b9033a8e29a9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 66, 'total_tokens': 138, 'input_token_details': {}, 'output_token_details': {}})

In [25]:
# Getting the final response output
response['messages'][-1].content

'The capital of France is **Paris**. It’s not only the political center but also a major hub for culture, art, fashion, and history.'

Just as before, we can use system messages to provide higher level instructions/guidance to the LLM.

In [26]:
response = agent.invoke(
    {'messages': [SystemMessage(content='write one-word responses'),
                  HumanMessage(content='What is the capital of france?')]}
)

In [27]:
response

{'messages': [SystemMessage(content='write one-word responses', additional_kwargs={}, response_metadata={}, id='6dba12ca-61a5-493d-b357-f1307e3fc9f2'),
  HumanMessage(content='What is the capital of france?', additional_kwargs={}, response_metadata={}, id='d70c71e7-beed-469d-95cc-d7af342d169c'),
  AIMessage(content='Paris', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 80, 'total_tokens': 138, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-a4f4ad61-7566-4bac-8958-8b1ccf3575c0', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d26-7ca4-7731-96e6-c3cb37c54303-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 58, 'total_tokens': 138, 'input_token_details': {}, 'output_token_details': {}})]}

In [28]:
# Getting the final response output
response['messages'][-1].content

'Paris'

By the way, this does also work for `model.invoke` (as opposed to `agent.invoke`) but **not** in the same way

In [29]:
# will give an error
response = model.invoke(
    {'messages': [SystemMessage(content='write one-word responses'),
                  HumanMessage(content='What is the capital of france?')]}
)

ValueError: Invalid input type <class 'dict'>. Must be a PromptValue, str, or list of BaseMessages.

In [30]:
# will NOT give an error
response = model.invoke(
    [SystemMessage(content='write one-word responses'),
     HumanMessage(content='What is the capital of france?')]
)

In [31]:
response

AIMessage(content='Paris', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 80, 'total_tokens': 124, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-c9597f45-5863-4e77-83d1-134675e2660e', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c5d26-9153-7911-9507-6e19cd46d30b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 44, 'total_tokens': 124, 'input_token_details': {}, 'output_token_details': {}})

Turning back to `agent`, there are some useful things to get started on for conversations and agent use.

We can chain messages to create something like a conversation.

In [32]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of CA?"),
                  AIMessage(content="The capital of CA is Los Angeles."),
                  HumanMessage(content="Interesting, how did Los Angeles become the capital?")]}
)

In [33]:
response['messages'][-1].content

'I’m sorry about the mistake in my earlier reply—**the capital of California is Sacramento**, not Los Angeles.  \n\nBelow is a brief overview of how Sacramento came to be the state’s capital and why Los Angeles never held that title.\n\n---\n\n## 1. Early Statehood and the Search for a Capital  \n\n| Year | Event |\n|------|-------|\n| **1849** | California’s **Constitutional Convention** meets in **Monterey** (the first provisional capital). |\n| **1849–1850** | The **first elected legislature** convenes in **San Jose**, which serves as the temporary capital while a permanent location is chosen. |\n| **1850** | California is admitted to the Union on **September\u202f9**. The legislature must decide on a permanent capital. |\n\n## 2. Why Sacramento Was Chosen  \n\n1. **Geographic Centrality (for the time)**  \n   - In the 1850s, most of California’s population and economic activity were concentrated in the **Northern and Central** parts of the state, especially around the Sierra Nevada

In [34]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
                  AIMessage(content="The capital of the Moon is Luna City."),
                  HumanMessage(content="Interesting, tell me more about Luna City")]}
)

In [35]:
from IPython.display import display, Markdown

In [36]:
text = response['messages'][-1].content
display(Markdown(text))

### Luna City – A (Fictional) Overview

> **Quick facts**  
> **Location:** Mare Tranquillitatis (the “Sea of Tranquility”) on the near‑side of the Moon  
> **Founded:** 2035 (first permanent settlement) – declared “capital” in 2048  
> **Population (2025 estimate):** ~12,000 permanent residents, plus rotating crews, tourists, and research teams  
> **Government:** Lunar Commonwealth Council (elected by all resident citizens, with representation for scientific, commercial, and indigenous‑heritage sectors)  
> **Official language(s):** English (primary), plus a growing “Lunish” pidgin of English, Russian, Mandarin, and Arabic technical terms  
> **Time‑keeping:** Luna Standard Time (LST) – 27.3 Earth days per lunar day; the city runs on a 24‑hour “Earth‑aligned” schedule for coordination with Earth‑based partners  

---

## 1. Why “Capital”?

In the speculative timeline that imagines a thriving lunar civilization, Luna City earned the title **“capital”** because it became the political, cultural, and logistical hub of the **Lunar Commonwealth**—a loose federation of settlements, research outposts, and commercial bases spread across the Moon’s near side. The city hosts the **Lunar Commonwealth Council**, the main legislative body that drafts inter‑settlement law, coordinates resource allocation, and negotiates with Earth governments and corporations.

---

## 2. Geography & Layout

| Feature | Description |
|---------|-------------|
| **Mare Tranquillitatis** | A basalt‑filled plain that offers a relatively smooth, low‑gradient surface—ideal for large‑scale construction and rover traffic. |
| **Regolith Shielding** | Most structures are buried 2–3 m beneath the surface, using compacted regolith as radiation and micrometeorite shielding. |
| **Central Dome (The Atrium)** | A 1‑km‑diameter transparent‑aluminum dome that encloses the civic center, parks, and the “Lunar Plaza.” Inside, a controlled atmosphere mimics Earth‑like temperature (≈22 °C) and pressure (101 kPa). |
| **Habitat Modules** | Radiant‑glass “spider‑web” habitats radiate outward from the Atrium, each housing 200–500 residents. They are connected by pressurized tunnels and magnetic‑levitation (mag‑lev) trams. |
| **Industrial Zone** | Located on the western edge, it contains oxygen‑extraction plants (MOXIE‑type), 3‑D‑printed construction yards, and a small “lunar‑brewery” that uses regolith‑derived minerals for fermentation. |
| **Science & Exploration Hub** | Adjacent to the industrial zone, this area houses the **Lunar Research Institute (LRI)**, a launch pad for micro‑launchers, and a “Moon‑Mars Transfer Port” for deep‑space missions. |
| **Cultural Quarter** | Features the **Luna Gallery**, a holographic theatre, a multilingual library, and the “Moon‑Market” where artisans sell handcrafted regolith‑ceramics, solar‑fabric clothing, and “lunar‑coffee.” |

---

## 3. History in a Nutshell

| Year | Milestone |
|------|-----------|
| **2035** | First permanent habitat (Habitat‑Alpha) erected by the **International Lunar Initiative (ILI)**. |
| **2039** | First commercial mining operation (Regolith‑Extract Ltd.) begins extracting water ice from permanently shadowed craters, transporting it to Luna City via autonomous rovers. |
| **2042** | The **Lunar Charter** is signed by 12 Earth nations, establishing a shared governance model for lunar activities. |
| **2045** | Construction of the Atrium dome begins; the city’s population surpasses 5,000. |
| **2048** | The **Lunar Commonwealth Council** convenes for the first time; Luna City is officially designated the capital. |
| **2052** | First “Lunar Olympics” held, featuring low‑gravity sports like “Moon‑ball” and “Regolith‑Racing.” |
| **2058** | Luna City’s **Solar‑Array Farm** reaches 200 MW, making the city energy‑independent and a net exporter of surplus power to other lunar bases. |
| **2065** | The city celebrates its 30th anniversary with the unveiling of the **Luna Spiral**, a 2‑km‑long, spiraling park that doubles as a thermal regulator for the underlying regolith. |

---

## 4. Governance & Society

### 4.1 Lunar Commonwealth Council
- **Structure:** 27 seats, proportional representation based on settlement population and functional sectors (science, industry, tourism, heritage).  
- **Term:** 4 Earth‑years, with staggered elections to ensure continuity.  
- **Key Powers:** Approve inter‑settlement trade agreements, allocate shared resources (e.g., water, power), set safety standards for EVA (extravehicular activity) protocols, and negotiate Earth‑Moon treaties.

### 4.2 Civic Life
- **Education:** The **Luna Academy** offers K‑12 schooling with a curriculum that blends Earth subjects with lunar‑specific courses (e.g., regolith engineering, low‑gravity biomechanics).  
- **Healthcare:** A tele‑medicine network links Luna City’s **Lunar Medical Center** to Earth hospitals; on‑site surgeons are trained for both Earth‑like and low‑gravity surgeries.  
- **Culture:** Residents celebrate **“First Step Day”** (July 20) and **“Lunar Harvest”** (the day the solar array farm reaches peak output). Music, art, and literature often explore themes of isolation, Earth‑longing, and the beauty of the star‑filled sky.

### 4.3 Legal Framework
- **Criminal Law:** Based on a hybrid of Earth common law and a **Lunar Code of Conduct** that emphasizes resource stewardship and safety.  
- **Property Rights:** Surface rights are managed by the **Lunar Land Registry**, which issues “use‑leases” rather than outright ownership, reflecting the principle that the Moon is a shared commons.

---

## 5. Economy & Infrastructure

| Sector | Highlights |
|--------|------------|
| **Energy** | 200 MW of solar‑array farms (thin‑film panels on the far side of the dome) + 30 MW of small nuclear fission units (used as backup). |
| **Water & Oxygen** | Water ice harvested from Shackleton Crater is processed in the **Luna Waterworks**; oxygen is split via electrolysis and stored in underground tanks. |
| **Manufacturing** | 3‑D‑printing of habitat components, antennae, and spare parts using regolith‑derived “lunar concrete.” |
| **Tourism** | ~1,200 tourists per year (as of 2025) who stay in the **Luna Lodge**, a pressurized hotel with a transparent dome offering 360° views of Earth. |
| **Research** | The **Lunar Research Institute** runs experiments in low‑gravity biology, plasma physics, and in‑situ resource utilization (ISRU). |
| **Transport** | Mag‑lev trams (max 150 km/h) connect the city’s districts; autonomous rovers ferry cargo to outlying mining sites; a small launch pad supports “lunar‑to‑Mars” cargo rockets. |

---

## 6. Daily Life in Luna City

1. **Morning Routine** – Residents wake to a gentle amber glow from the Atrium’s artificial sunrise cycle (designed to mimic Earth’s circadian rhythm). Breakfast often includes “lunar‑brew” coffee (made from beans grown in hydroponic farms) and a protein‑rich algae smoothie.  
2. **Commute** – Most people ride the mag‑lev tram or walk through climate‑controlled tunnels. The low gravity (≈1/6 g) makes movement feel “bouncy,” so many use small, spring‑loaded shoes to absorb impact.  
3. **Work** – Jobs range from habitat maintenance, regolith processing, scientific research, to tourism guides who lead “low‑gravity tours” of the surrounding basalt plains.  
4. **Leisure** – In the evenings, families gather at the **Luna Plaza** for holographic concerts, or they drift in the **Zero‑G Playroom**, a large chamber where people can float and play low‑gravity sports.  
5. **Night** – The Atrium’s transparent dome offers an unobstructed view of Earth and the stars. Many residents spend time stargazing, using the city’s **Lunar Observatory** to track near‑Earth objects.

---

## 7. Challenges & Future Outlook

| Challenge | Current Mitigation | Future Plans |
|-----------|--------------------|--------------|
| **Radiation** | Regolith shielding + active magnetic field generators around critical facilities. | Expand magnetic shielding to cover the entire city perimeter; develop “radiation‑absorbing” nanomaterials for next‑gen habitats. |
| **Psychological Health** | Regular Earth‑contact via video, communal events, and “Earth‑day” celebrations. | Implement VR “nature immersion” suites that simulate forests, oceans, and open skies. |
| **Supply Chain Dependence** | On‑site ISRU reduces reliance on Earth shipments; 3‑D printing of spare parts. | Aim for 90% self‑sufficiency by 2070, including food production via vertical farms and algae bioreactors. |
| **Governance Coordination** | The Lunar Commonwealth Council works with Earth space agencies through the **International Lunar Governance Forum**. | Formalize a **Moon‑Earth Treaty** that codifies resource sharing, environmental protection, and dispute resolution. |
| **Economic Viability** | Tourism, research grants, and export of high‑purity oxygen and rare earths. | Develop a lunar “manufacturing hub” for ultra‑pure silicon wafers and quantum‑grade materials, leveraging the Moon’s low‑vibration environment. |

---

## 8. Fun Tidbits & Cultural Nuggets

- **Lunish Language:** A playful creole that mixes English with Russian, Mandarin, and Arabic technical terms. Example: “**Zdravstvyu, comrade!**” (Hello, friend!) is a common greeting.  
- **Luna Spiral Park:** The park’s spiraling pathways are lined with bioluminescent algae that glow softly at night, creating a “river of stars” effect.  
- **Moon‑Mars Transfer Port:** The port’s launch pad is designed for “hopper” rockets that can quickly ferry cargo from Luna City to a staging orbit, then onward to Mars.  
- **Lunar Olympics:** Sports include “Moon‑ball” (a low‑gravity version of volleyball with a heavier ball) and “Regolith‑Racing,” where participants navigate a course on wheeled rovers while performing acrobatic jumps.  
- **Luna City’s Flag:** A white circle (the Moon) on a dark blue field, with a stylized silver “L” representing the Atrium dome and the city’s commitment to light and exploration.

---

### TL;DR

Luna City is a speculative, self‑sustaining lunar metropolis perched in Mare Tranquillitatis. It serves as the political capital of a fledgling Lunar Commonwealth, blending cutting‑edge engineering (buried habitats, solar farms, mag‑lev transit) with a vibrant, Earth‑connected culture. While challenges like radiation and supply‑chain dependence remain, the city’s roadmap aims for near‑complete self‑sufficiency, deeper scientific discovery, and a thriving community that looks both back to Earth and forward to the rest of the solar system.

Too much info??

## Streaming example

`agent.stream` will allow us to stream the output instead of getting a massive string all at once.

Specifically, it will return two outputs in chunked intervals, with the first being a message chunk that contains response content.

In [37]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token

## Luna City – The “Capital” of the Moon (A 21st‑century speculative overview)

> **TL;DR** – Luna City is a multi‑national, partially‑private lunar settlement that has grown from a handful of scientific outposts into the Moon’s first true “city‑state.”  It sits in the permanently‑shadowed region of Shackleton Crater near the lunar south pole, where water‑ice, near‑continuous sunlight on the rim, and a relatively benign terrain make it the most logical place for a permanent capital.  Its governance is a hybrid United Nations‑backed Lunar Authority (ULA) with a city council elected by residents, while major infrastructure is owned and operated by a consortium of space‑faring corporations.  The city’s economy revolves around helium‑3 mining, in‑situ resource utilization (ISRU), high‑value manufacturing, scientific research, and tourism.  Life is a blend of high‑tech habitats, underground regolith shielding, and a vibrant low‑gravity culture.

---

## 1.  Why “Luna City” Exists – The Hist

# Having conversations with memory

Another useful conversational element is retaining memory of content.  ConversationBufferMemory and etc have been deprecated, but a currently useful way to do this is with LangGraph's InMemoryStore.



In [38]:
question = HumanMessage(content="My name is Ben and my favourite color is aqua.")

response = agent.invoke(
    {"messages": [question]} 
)

response['messages'][-1].content

'Nice to meet you, Ben! 🌊 Aqua is such a refreshing, calming shade—great choice. If there’s anything you’d like to chat about, explore, or need help with, just let me know!'

In [39]:
question = HumanMessage(content="What is my favourite color?")

response = agent.invoke(
    {"messages": [question]} 
)

response['messages'][-1].content

'I’m not sure—your favorite color isn’t something I can read from here. 😊  \n\nWhat color do you love most? Feel free to share, and we can chat about why it might be your favorite!'

In [40]:
from langgraph.checkpoint.memory import InMemorySaver  

We need to initialize the model with checkpointer.

In [41]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  
)

We also need to specify a thread ID number so that the LLM can keep track of a continous discussion thread.

In [42]:
question = HumanMessage(content="My name is Ben and my favourite color is aqua.")

response = agent.invoke(
    {"messages": [question]},
    {"configurable": {"thread_id": "1"}}
)

response['messages'][-1].content

'Nice to meet you, Ben! 🎉  \nAqua is such a refreshing, calming shade—great choice. If there’s anything you’d like to chat about, need help with, or just want to share more of your favorite things, feel free to let me know!'

In [43]:
question = HumanMessage(content="What is my favourite color?")

response = agent.invoke(
    {"messages": [question]},
    {"configurable": {"thread_id": "1"}}
)

response['messages'][-1].content

'Your favourite colour is **aqua**. 🌊'

For production use cases, it's better to use a persistent store such as a database.